# Lesson 5: Batches (chunks of text + their answers)

**Why:** the model can't read 976,605 characters at once. We feed it **short chunks** and ask it, at every position, *"what comes next?"* The answers are free: they're just the same text **shifted by one character**.

```
chunk  x:   T   h   e   ␣   c   u   r   r
answer y:   h   e   ␣   c   u   r   r   e      <- each x character's "next character"
```

The chunk length is called **`block_size`** (our **T**). A **batch** is several random chunks stacked together (our **B**), so the model learns from many places in the book at once.

## 1. Load the text, and split off an "exam" part

We reuse Lesson 4's tokenizer. Then we split the book:
- **90% training data:** the model learns from this.
- **10% validation data:** the model **never** trains on this. We use it to check whether the model truly learned Tesla's style, or just memorized the training text. Think of it as exam questions that weren't in the homework.

In [ ]:
from pathlib import Path
import torch

torch.manual_seed(42)

text = Path("../data/input.txt").read_text(encoding="utf-8")
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"training:   {len(train_data):,} characters")
print(f"validation: {len(val_data):,} characters")

## 2. One chunk hides many examples

We take one chunk of `block_size = 8` characters, plus its answers shifted by one.

**Predict:** how many separate "given this, what comes next?" examples does one chunk of 8 characters contain?

In [ ]:
block_size = 8
start = 1000                                  # some place in the book
x = train_data[start : start + block_size]
y = train_data[start + 1 : start + block_size + 1]

print("x:", repr(decode(x.tolist())))
print("y:", repr(decode(y.tolist())))
print()
for t in range(block_size):
    context = decode(x[: t + 1].tolist())
    target = decode([y[t].item()])
    print(f"given {context!r:14} -> next is {target!r}")

## 3. A batch: several random chunks at once

`get_batch` picks **B** random starting points, cuts a chunk from each, and stacks them into a grid. This is the function that feeds the GPT during training.

(`torch.randint` picks random whole numbers; `torch.stack` piles equal-length rows into one tensor.)

In [ ]:
batch_size = 4   # B

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])          # (B, T)
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])  # (B, T)
    return x, y

xb, yb = get_batch("train")
print("x shape:", xb.shape, "  y shape:", yb.shape)
print()
for b in range(batch_size):
    print(f"chunk {b}:  x={decode(xb[b].tolist())!r:12}  y={decode(yb[b].tolist())!r}")

## 4. Final check

The key property: **y is x shifted left by one**. Everything in x from position 1 onward must equal y up to its last position.

In [ ]:
assert xb.shape == yb.shape == (batch_size, block_size)
assert torch.equal(yb[:, :-1], xb[:, 1:])
print(f"One batch = {batch_size} chunks x {block_size} positions = {batch_size * block_size} training examples.")
print("Lesson 5 complete.")

## Wrap-up question

With `block_size = 8`, what is the **most** text the model can look back at when it predicts the next character? What problem could that cause when writing about something like "alternating current"?